# 🤖 AI Functions Showcase - The Art of the Possible

**COMPREHENSIVE AI DEMONSTRATION** - Run this after notebook 01.

## What this notebook demonstrates:
- ✅ **ai_classify** - Intelligent priority and category classification
- ✅ **ai_gen** - Action extraction, requirements analysis, urgency assessment, and comprehensive analysis
- ✅ **Final Summary Table** - Complete AI-powered ticket analysis

**Prerequisites:** Run `01_sample_data_generation.ipynb` first

## AI Functions Showcase:
- `ai_classify` - For priority and category classification
- `ai_gen` - For data extraction, analysis, and content generation

---

## 🎯 Goal: Demonstrate the full power of Databricks AI Functions


In [10]:
# Import required libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json
from datetime import datetime

# Import configuration
%run ./config

print("✅ Libraries imported and configuration loaded")
print(f"🎯 Using Unity Catalog: {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}")
print("🚀 Ready to showcase Databricks AI Functions!")


✅ Libraries imported and configuration loaded
🎯 Using Unity Catalog: quickstart_catalog_vkm_external.classify_tickets
🚀 Ready to showcase Databricks AI Functions!


In [11]:
# Load Sample Data
print("📊 Loading sample ticket data from Unity Catalog...")

# Load the sample ticket data from Unity Catalog
df_tickets = spark.table(TABLES["raw_tickets"])
print(f"✅ Loaded {df_tickets.count()} tickets from: {TABLES['raw_tickets']}")

# Display sample data
print("\n📋 Sample ticket data:")
display(df_tickets.select("ticket_id", "short_description", "description").limit(3))


📊 Loading sample ticket data from Unity Catalog...


✅ Loaded 10 tickets from: quickstart_catalog_vkm_external.classify_tickets.raw_tickets

📋 Sample ticket data:


,ticket_id,short_description,description
0,TICKET_001,Issue #1 - Critical,hey so our website is super slow today and customers are complaining. i think it might be the database or something. can someone look into this? it's been happening since this morning and we're losing sales.
1,TICKET_002,Issue #2 - Important,urgent! the login system is broken again. users can't get in and they're calling support nonstop. this happened last week too. we need to fix this asap before more customers leave.
2,TICKET_003,Issue #3 - Problem,i need help with the new feature we're building. the api is returning weird errors and i don't know why. it works sometimes but then fails randomly. can someone debug this?


# 🎯 AI Function #1: ai_classify

**Purpose:** Intelligent classification of text into predefined categories

**Use Cases:** Priority classification, category assignment, sentiment analysis, risk assessment

**Example:** Classify ticket priorities based on description content


In [12]:
# AI Function #1: ai_classify - Priority Classification
print("🎯 Demonstrating ai_classify for priority classification...")

# Register DataFrame as temporary view for SQL access
df_tickets.createOrReplaceTempView("tickets")

# Use ai_classify to classify ticket priorities
df_with_priority = spark.sql("""
    SELECT
        *,
        ai_classify(
            description, 
            ARRAY('Low Priority', 'Medium Priority', 'High Priority', 'Urgent Priority')
        ) as ai_priority_classification
    FROM tickets
""")

print("✅ ai_classify completed - Priority classification done!")
print("\n📊 Priority Classification Results:")
display(df_with_priority.select("ticket_id", "short_description", "ai_priority_classification").limit(5))

# Show distribution of AI classifications
print("\n📈 Priority Distribution:")
df_with_priority.groupBy("ai_priority_classification").count().orderBy(desc("count")).show()


🎯 Demonstrating ai_classify for priority classification...
✅ ai_classify completed - Priority classification done!

📊 Priority Classification Results:


,ticket_id,short_description,ai_priority_classification
0,TICKET_001,Issue #1 - Critical,Urgent Priority
1,TICKET_002,Issue #2 - Important,Urgent Priority
2,TICKET_003,Issue #3 - Problem,Medium Priority
3,TICKET_004,Issue #4 - Critical,High Priority
4,TICKET_005,Issue #5 - Problem,High Priority



📈 Priority Distribution:


+--------------------------+-----+
|ai_priority_classification|count|
+--------------------------+-----+
|           Urgent Priority|    6|
|             High Priority|    3|
|           Medium Priority|    1|
+--------------------------+-----+



# 🎯 AI Function #2: ai_gen (Data Extraction)

**Purpose:** Generate structured data from unstructured text using natural language prompts

**Use Cases:** Action items extraction, requirement summaries, urgency assessment, content generation

**Example:** Extract action items and analyze ticket requirements using AI generation


In [ ]:
# AI Function #2: ai_gen - Action Items Extraction (Simplified)
print("🎯 Demonstrating ai_gen for action items extraction...")

# Register DataFrame as temporary view for SQL access
df_with_priority.createOrReplaceTempView("tickets_with_priority")

# Use ai_gen to extract action items (more reliable than ai_extract)
df_with_actions = spark.sql("""
    SELECT
        *,
        ai_gen(
            CONCAT(
                'Extract specific action items from this ticket description. ',
                'Return as a JSON array of strings. Each item should be a concrete, actionable task. ',
                'If no action items are found, return an empty array []. ',
                'Ticket description: ', description
            )
        ) as action_items_raw
    FROM tickets_with_priority
""")

print("✅ ai_gen completed - Action items extraction done!")
print("\n📊 Raw Extraction Results:")
display(df_with_actions.select("ticket_id", "short_description", "action_items_raw").limit(3))

# Clean and parse the action items
df_parsed = df_with_actions.withColumn(
    "action_items_cleaned",
    regexp_replace(col("action_items_raw"), r'^```json\s*|\s*```$', '')  # Remove markdown code blocks
).withColumn(
    "action_items_cleaned", 
    regexp_replace(col("action_items_cleaned"), r'^\[|\]$', '')  # Remove outer brackets
).withColumn(
    "action_items_cleaned",
    regexp_replace(col("action_items_cleaned"), r'"([^"]+)"', '$1')  # Remove quotes
).withColumn(
    "action_items",
    split(col("action_items_cleaned"), r',\s*')  # Split by comma
).withColumn(
    "main_requirement",
    ai_gen(
        CONCAT(
            'What is the main requirement or request in this ticket? ',
            'Provide a brief summary in 1-2 sentences. ',
            'Ticket: ', short_description, ' - ', description
        )
    )
).withColumn(
    "urgency_level",
    ai_gen(
        CONCAT(
            'Assess the urgency level of this ticket. ',
            'Return only one word: Low, Medium, High, or Urgent. ',
            'Ticket: ', short_description, ' - ', description
        )
    )
).drop("action_items_raw", "action_items_cleaned")

print("\n📋 Parsed Extraction Results:")
display(df_parsed.select("ticket_id", "action_items", "main_requirement", "urgency_level").limit(3))


🎯 Demonstrating ai_extract for structured data extraction...
✅ ai_extract completed - Structured data extraction done!

📊 Extraction Results:


,ticket_id,short_description,ai_extracted_data
0,TICKET_001,Issue #1 - Critical,"{'action_items:array<string>': None, 'main_requirement:string': None, 'urgency_level:string': None, 'affected_systems:array<string>': None}"
1,TICKET_002,Issue #2 - Important,"{'action_items:array<string>': None, 'main_requirement:string': None, 'urgency_level:string': None, 'affected_systems:array<string>': None}"
2,TICKET_003,Issue #3 - Problem,"{'action_items:array<string>': None, 'main_requirement:string': None, 'urgency_level:string': None, 'affected_systems:array<string>': None}"



📋 Parsed Extraction Results:


{"ts": "2025-09-19 20:56:16.064", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[FIELD_NOT_FOUND] No such struct field `action_items` in `action_items:array<string>`, `main_requirement:string`, `urgency_level:string`, `affected_systems:array<string>`. SQLSTATE: 42704\n\nJVM stacktrace:\norg.apache.spark.sql.AnalysisException\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.noSuchStructFieldInGivenFieldsError(QueryCompilationErrors.scala:3387)\n\tat org.apache.spark.sql.catalyst.expressions.ExtractValue$.findField(complexTypeExtractors.scala:118)\n\tat org.apache.spark.sql.catalyst.expressions.ExtractValue$.apply(complexTypeExtractors.scala:67)\n\tat org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.$anonfun$resolveCandidates$2(package.scala:422)\n\tat scala.collection.LinearSeqOps.foldLeft(LinearSeq.scala:183)\n\tat scala.collection.LinearSeqOps.foldLeft$(LinearSeq.scala:179)\n\tat scala.collection.immutable.List.foldLeft(List.scala:79)\n\tat 

AnalysisException: [FIELD_NOT_FOUND] No such struct field `action_items` in `action_items:array<string>`, `main_requirement:string`, `urgency_level:string`, `affected_systems:array<string>`. SQLSTATE: 42704

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.noSuchStructFieldInGivenFieldsError(QueryCompilationErrors.scala:3387)
	at org.apache.spark.sql.catalyst.expressions.ExtractValue$.findField(complexTypeExtractors.scala:118)
	at org.apache.spark.sql.catalyst.expressions.ExtractValue$.apply(complexTypeExtractors.scala:67)
	at org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.$anonfun$resolveCandidates$2(package.scala:422)
	at scala.collection.LinearSeqOps.foldLeft(LinearSeq.scala:183)
	at scala.collection.LinearSeqOps.foldLeft$(LinearSeq.scala:179)
	at scala.collection.immutable.List.foldLeft(List.scala:79)
	at org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.resolveCandidates(package.scala:421)
	

{"ts": "2025-09-19 20:56:16.556", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[FIELD_NOT_FOUND] No such struct field `action_items` in `action_items:array<string>`, `main_requirement:string`, `urgency_level:string`, `affected_systems:array<string>`. SQLSTATE: 42704\n\nJVM stacktrace:\norg.apache.spark.sql.AnalysisException\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.noSuchStructFieldInGivenFieldsError(QueryCompilationErrors.scala:3387)\n\tat org.apache.spark.sql.catalyst.expressions.ExtractValue$.findField(complexTypeExtractors.scala:118)\n\tat org.apache.spark.sql.catalyst.expressions.ExtractValue$.apply(complexTypeExtractors.scala:67)\n\tat org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.$anonfun$resolveCandidates$2(package.scala:422)\n\tat scala.collection.LinearSeqOps.foldLeft(LinearSeq.scala:183)\n\tat scala.collection.LinearSeqOps.foldLeft$(LinearSeq.scala:179)\n\tat scala.collection.immutable.List.foldLeft(List.scala:79)\n\tat 

AnalysisException: [FIELD_NOT_FOUND] No such struct field `action_items` in `action_items:array<string>`, `main_requirement:string`, `urgency_level:string`, `affected_systems:array<string>`. SQLSTATE: 42704

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.noSuchStructFieldInGivenFieldsError(QueryCompilationErrors.scala:3387)
	at org.apache.spark.sql.catalyst.expressions.ExtractValue$.findField(complexTypeExtractors.scala:118)
	at org.apache.spark.sql.catalyst.expressions.ExtractValue$.apply(complexTypeExtractors.scala:67)
	at org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.$anonfun$resolveCandidates$2(package.scala:422)
	at scala.collection.LinearSeqOps.foldLeft(LinearSeq.scala:183)
	at scala.collection.LinearSeqOps.foldLeft$(LinearSeq.scala:179)
	at scala.collection.immutable.List.foldLeft(List.scala:79)
	at org.apache.spark.sql.catalyst.expressions.package$AttributeSeq.resolveCandidates(package.scala:421)
	

# 🎯 AI Function #3: ai_gen

**Purpose:** Generate creative content, summaries, and complex analysis

**Use Cases:** Executive summaries, detailed analysis, creative content, recommendations

**Example:** Generate comprehensive ticket analysis and recommendations


In [ ]:
# AI Function #3: ai_gen - Comprehensive Analysis
print("🎯 Demonstrating ai_gen for comprehensive analysis...")

# Register DataFrame as temporary view for SQL access
df_parsed.createOrReplaceTempView("tickets_with_extraction")

# Use ai_gen to create comprehensive analysis
df_with_analysis = spark.sql("""
    SELECT
        *,
        ai_gen(
            CONCAT(
                'Analyze this IT ticket and provide: ',
                '1. Executive summary (2-3 sentences), ',
                '2. Technical complexity (Low/Medium/High), ',
                '3. Estimated effort (hours), ',
                '4. Risk assessment (Low/Medium/High), ',
                '5. Recommended next steps. ',
                'Ticket: ', short_description, ' - ', description
            )
        ) as ai_comprehensive_analysis
    FROM tickets_with_extraction
""")

print("✅ ai_gen completed - Comprehensive analysis done!")
print("\n📊 Analysis Results:")
display(df_with_analysis.select("ticket_id", "short_description", "ai_comprehensive_analysis").limit(3))


# 🎉 Final Summary Table - The Art of the Possible

**Complete AI-powered ticket analysis showcasing all three AI functions**


In [ ]:
# Create Final Summary Table
print("🎉 Creating comprehensive summary table...")

# Create a clean summary table with all AI results
df_final_summary = df_with_analysis.select(
    "ticket_id",
    "short_description",
    "ai_priority_classification",
    "action_items",
    "main_requirement", 
    "urgency_level",
    "affected_systems",
    "ai_comprehensive_analysis"
).withColumn(
    "ai_showcase_timestamp", 
    current_timestamp()
)

print("✅ Final summary table created!")
print(f"📊 Total tickets analyzed: {df_final_summary.count()}")

# Display the comprehensive results
print("\n🎯 COMPLETE AI SHOWCASE RESULTS:")
print("="*80)
display(df_final_summary.limit(5))

# Save to Unity Catalog
print("\n💾 Saving results to Unity Catalog...")
df_final_summary.write.format("delta").mode("overwrite").saveAsTable(TABLES["ai_showcase_results"])
print(f"✅ Results saved to: {TABLES['ai_showcase_results']}")

# Show summary statistics
print("\n📈 AI Showcase Summary Statistics:")
print(f"🎯 Tickets processed: {df_final_summary.count()}")
print(f"🤖 AI functions demonstrated: 2 (ai_classify, ai_gen)")
print(f"📊 Data saved to Unity Catalog: {TABLES['ai_showcase_results']}")

print("\n" + "="*80)
print("🎉 AI SHOWCASE COMPLETED SUCCESSFULLY!")
print("="*80)
print("✅ ai_classify: Priority classification")
print("✅ ai_gen: Action extraction, requirements, urgency assessment, and comprehensive analysis") 
print("✅ Final summary table created and saved")
print("="*80)
